## Step 8 — assign population to candidate new blocks
**# of cells in notebook:** 1

**Purpose:** Assign population to each candidate new block layer created in Step 7. Population from the source population grid is apportioned among candidate blocks according to the share of building footprint area falling within each population-grid-cell/block combination.

**Input:**

- `new_blocks.gpkg` from Step 7 for each selected source block
- a population geodatabase containing:
  - `pop_grid`
  - `buildings_inside`
- `heterogeneous_largePop_blocks` from Step 1, used to recover the original screening status of each source block

**Output:**

Within each block folder:

- `new_blocks_populated.gpkg` — the Step 7 candidate layers with a `population` field
- `<blk_layer>_assigned_population.csv`
- `<blk_layer>_grid_apportionment_detail.csv`

At the base block directory:

- `new_blocks_population_assignment_summary_no_arcpy.csv`
- `new_blocks_population_assignment_no_arcpy.log`

The populated layers also receive fields describing whether the original source block was heterogeneous and/or LargePop.

**Main logic:**

**Cell 1 — Allocate population using building-area shares**

1. Reads all candidate `blk_*` layers from `new_blocks.gpkg` and looks up the screening flags of the original source block.
2. Selects population-grid cells and buildings relevant to the candidate block extent.
3. Intersects buildings with population-grid cells and calculates the total building area within each grid cell.
4. Intersects those grid/building pieces with the candidate block polygons and calculates the building area belonging to each candidate block.
5. Calculates each candidate block's share of building area within each grid cell and multiplies that share by the grid-cell population.
6. Sums the apportioned population across grid cells for each candidate block.
7. Adds `population`, `hetero_orig`, and `large_pop_orig` to the candidate block layers.
8. Writes populated layers together with detailed allocation and summary CSVs.


In [ ]:
# -*- coding: utf-8 -*-
r"""
Assign population to new block layers WITHOUT ArcPy.

GeoPandas / pyogrio version of the population-assignment workflow.

Inputs:
    <block_folder>\new_blocks.gpkg
        blk_1_<source_block>_2
        blk_1_<source_block>_3
        blk_1_<source_block>_4
        blk_1_<source_block>_5

    E:\World Bank deliverbale 1\_analysis\population\population.gdb
        pop_grid
        buildings_inside

Outputs:
    <block_folder>\new_blocks_populated.gpkg
        blk_* layers with a population field

    Per-layer CSVs:
        <blk_layer>_assigned_population.csv
        <blk_layer>_grid_apportionment_detail.csv

    Overall CSV:
        new_blocks_population_assignment_summary_no_arcpy.csv

Important:
    - No arcpy.
    - Population inputs remain in Esri FileGDB.
"""

import re
import csv
import traceback
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import pandas as pd
import geopandas as gpd
import pyogrio


# ============================================================
# USER SETTINGS
# ============================================================

large_pop_blocks_folder = Path(
    r"E:\_johannesburg\_analysis\heterogeneous_largePop_blocks"
)

population_gdb = Path(
    r"E:\_johannesburg\_analysis\population_wp2\population_wp2.gdb"
)

# Source block membership lookup.
# This tells us whether each original block was heterogeneous, LargePop, both, or neither.
source_blocks_gdb = Path(
    r"E:\_johannesburg\_analysis\blocks\blocks.gdb"
)
source_blocks_layer = "heterogeneous_largePop_blocks"

pop_grid_layer = "pop_grid"
buildings_layer = "buildings_inside"

new_blocks_gpkg_name = "new_blocks.gpkg"
populated_blocks_gpkg_name = "new_blocks_populated.gpkg"

population_field = "population"
area_field = "area_m_utm_new"
pop_field = "grid_code"

source_block_id_field = "block_id"
hetero_flag_fields = ["HH_CC", "HH_Grtr10ha", "CC_Grtr10ha"]
largepop_flag_field = "LargePop"

overwrite_outputs = True
fix_invalid_geometries = False
area_tolerance = 1e-9

# Logging
log_file = large_pop_blocks_folder / "new_blocks_population_assignment_no_arcpy.log"

# If True, every log message is also printed to the console.
# If False, only high-level messages are printed to the console.
verbose_console = False


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def msg(text="", also_console=None):
    """
    Write progress to a log file and optionally to the console.

    This is safer than printing thousands of lines into a Jupyter notebook output
    cell. It is also useful when running the script from Anaconda Prompt.

    Set verbose_console=True in USER SETTINGS if you want every message printed.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {text}"

    try:
        with open(log_file, "a", encoding="utf-8") as log:
            log.write(line + "\n")
    except Exception:
        # Last-resort fallback. Logging should never crash the workflow.
        pass

    if also_console is None:
        also_console = verbose_console

    if also_console:
        print(text, flush=True)


def safe_float(value):
    if value is None:
        return 0.0
    try:
        if pd.isna(value):
            return 0.0
    except Exception:
        pass
    try:
        return float(value)
    except Exception:
        return 0.0


def sanitize_name(name):
    name = re.sub(r"[^A-Za-z0-9_]", "_", name)
    if re.match(r"^[0-9]", name):
        name = "x_" + name
    return name


def get_k_suffix(layer_name):
    parts = layer_name.split("_")
    last = parts[-1]
    if last.isdigit():
        return f"k{last}"
    return "kall"


def list_layer_names(dataset_path):
    layers = pyogrio.list_layers(str(dataset_path))
    if hasattr(layers, "shape"):
        return [str(row[0]) for row in layers]
    return [str(row[0]) if isinstance(row, (list, tuple)) else str(row) for row in layers]


def layer_exists(dataset_path, layer_name):
    lower = layer_name.lower()
    return lower in {lyr.lower() for lyr in list_layer_names(dataset_path)}


def actual_layer_name(dataset_path, layer_name):
    lower = layer_name.lower()
    for lyr in list_layer_names(dataset_path):
        if lyr.lower() == lower:
            return lyr
    raise RuntimeError(f"Layer not found: {layer_name} in {dataset_path}")


def require_columns(df, required_columns, label):
    existing_lower = {c.lower() for c in df.columns}
    missing = [c for c in required_columns if c.lower() not in existing_lower]
    if missing:
        raise RuntimeError(f"{label} is missing required column(s): {missing}")


def read_vector(path, layer, columns=None, bbox=None, read_geometry=True):
    kwargs = {
        "layer": layer,
        "columns": columns,
        "bbox": bbox,
        "read_geometry": read_geometry,
    }
    kwargs = {k: v for k, v in kwargs.items() if v is not None}

    try:
        return pyogrio.read_dataframe(str(path), fid_as_index=True, **kwargs)
    except TypeError:
        return pyogrio.read_dataframe(str(path), **kwargs)


def add_stable_id_from_index(gdf, id_field):
    """
    Robustly creates an ID field from the current dataframe index.

    The first script assumed reset_index() would create a column named 'index'.
    But if pyogrio returns a named index such as 'fid' or 'OBJECTID', reset_index()
    creates that named column instead. This helper avoids that problem.
    """
    out = gdf.copy()
    out[id_field] = out.index.to_series(index=out.index).astype(str).values
    out = out.reset_index(drop=True)
    return out


def geometry_union(gdf):
    try:
        return gdf.geometry.union_all()
    except Exception:
        return gdf.geometry.unary_union


def make_valid_if_requested(gdf, label):
    if not fix_invalid_geometries:
        return gdf
    msg(f"    Repairing invalid geometries: {label}")
    out = gdf.copy()
    out["geometry"] = out.geometry.make_valid()
    return out


def is_projected_crs(gdf):
    try:
        return bool(gdf.crs and gdf.crs.is_projected)
    except Exception:
        return False


def ensure_same_crs(gdf, target_crs, label):
    if gdf.crs is None:
        raise RuntimeError(f"{label} has unknown CRS.")
    if target_crs is None:
        raise RuntimeError("Target layer has unknown CRS.")
    if gdf.crs != target_crs:
        msg(f"    Reprojecting {label} to match block layer CRS.")
        return gdf.to_crs(target_crs)
    return gdf


def write_gpkg_layer(gdf, gpkg_path, layer_name):
    gdf.to_file(
        str(gpkg_path),
        layer=layer_name,
        driver="GPKG",
        engine="pyogrio",
    )


def iter_block_folders(base_folder):
    for item in sorted(base_folder.iterdir()):
        if item.is_dir() and item.name.startswith("_"):
            yield item


def get_blk_layers(new_blocks_gpkg):
    layers = [
        lyr for lyr in list_layer_names(new_blocks_gpkg)
        if lyr.lower().startswith("blk_")
    ]
    return sorted(layers)


def read_block_layers_for_folder(new_blocks_gpkg, layer_names):
    block_layers = {}

    for layer_name in layer_names:
        gdf = read_vector(new_blocks_gpkg, layer=layer_name)

        if gdf.empty:
            msg(f"  Warning: {layer_name} is empty.")

        gdf = gdf.reset_index(drop=True).copy()
        gdf["_block_fid"] = range(1, len(gdf) + 1)

        block_layers[layer_name] = gdf

    return block_layers


def combined_bounds(block_layers):
    bounds = []
    for gdf in block_layers.values():
        if not gdf.empty:
            bounds.append(gdf.total_bounds)

    if not bounds:
        return None

    b = pd.DataFrame(bounds, columns=["minx", "miny", "maxx", "maxy"])
    return (
        float(b["minx"].min()),
        float(b["miny"].min()),
        float(b["maxx"].max()),
        float(b["maxy"].max()),
    )



def block_folder_to_source_block_id(block_folder_name):
    """
    Converts folder names like '_14' to source block IDs like 'blk_14',
    matching the block_id values in heterogeneous_largePop_blocks.
    """
    return "blk_" + block_folder_name.lstrip("_")


def flag_is_one(value):
    """
    Robust test for 1-valued flags that may be read as int, float, string, etc.
    """
    try:
        return int(float(value)) == 1
    except Exception:
        return False


def build_hetero_value(row):
    """
    Build hetero_orig from HH_CC, HH_Grtr10ha, and CC_Grtr10ha.

    Normally only one of these should be 1. If multiple are unexpectedly 1,
    this preserves the information by joining them with '|'.
    """
    active = []

    for field in hetero_flag_fields:
        if field in row.index and flag_is_one(row[field]):
            active.append(field)

    if not active:
        return "none"

    return "|".join(active)


def read_source_block_membership_lookup():
    """
    Reads the source block membership layer and returns a lookup:

        block_id -> {"hetero_orig": ..., "large_pop_orig": 0/1}
    """
    msg("Reading source block membership lookup...", also_console=True)
    msg(f"  Source GDB:   {source_blocks_gdb}", also_console=True)
    msg(f"  Source layer: {source_blocks_layer}", also_console=True)

    if not source_blocks_gdb.exists():
        raise FileNotFoundError(f"Source blocks GDB does not exist:\n{source_blocks_gdb}")

    if not layer_exists(source_blocks_gdb, source_blocks_layer):
        raise FileNotFoundError(
            f"Source blocks layer '{source_blocks_layer}' does not exist in:\n"
            f"{source_blocks_gdb}"
        )

    actual_layer = actual_layer_name(source_blocks_gdb, source_blocks_layer)
    required_cols = [source_block_id_field] + hetero_flag_fields + [largepop_flag_field]

    source_df = read_vector(
        source_blocks_gdb,
        layer=actual_layer,
        columns=required_cols,
        read_geometry=False,
    )

    require_columns(source_df, required_cols, "heterogeneous_largePop_blocks")

    lookup = {}
    duplicate_count = 0

    for _, row in source_df.iterrows():
        block_id = str(row[source_block_id_field]).strip()

        if not block_id:
            continue

        if block_id in lookup:
            duplicate_count += 1

        lookup[block_id] = {
            "hetero_orig": build_hetero_value(row),
            "large_pop_orig": 1 if flag_is_one(row[largepop_flag_field]) else 0,
        }

    msg(f"  Source block records read: {len(source_df):,}", also_console=True)
    msg(f"  Source block lookup keys:  {len(lookup):,}", also_console=True)

    if duplicate_count:
        msg(
            f"  WARNING: {duplicate_count:,} duplicate block_id value(s) found in source lookup; last value kept.",
            also_console=True,
        )

    return lookup


def get_origin_membership_for_folder(block_folder_name, source_lookup):
    """
    Returns origin membership for one block folder.

    If the block is missing from the lookup, returns conservative defaults:
        hetero_orig='none'
        large_pop_orig=0
    """
    source_block_id = block_folder_to_source_block_id(block_folder_name)

    if source_block_id not in source_lookup:
        msg(
            f"  WARNING: {source_block_id} not found in source membership lookup. "
            "Using hetero_orig='none' and large_pop_orig=0.",
            also_console=True,
        )
        return {
            "source_lookup_block_id": source_block_id,
            "hetero_orig": "none",
            "large_pop_orig": 0,
        }

    values = dict(source_lookup[source_block_id])
    values["source_lookup_block_id"] = source_block_id

    msg(
        f"  Source membership: {source_block_id} | "
        f"hetero_orig={values['hetero_orig']} | "
        f"large_pop_orig={values['large_pop_orig']}",
        also_console=True,
    )

    return values


def read_population_candidates(folder_block_layers):
    first_gdf = next(iter(folder_block_layers.values()))
    target_crs = first_gdf.crs

    if target_crs is None:
        raise RuntimeError("New block layer has unknown CRS.")

    bbox = combined_bounds(folder_block_layers)
    if bbox is None:
        raise RuntimeError("No non-empty block layers found for folder.")

    msg("  Reading candidate pop_grid cells from FileGDB...")
    msg(f"    bbox: {bbox}")

    pop_layer_actual = actual_layer_name(population_gdb, pop_grid_layer)

    pop_candidates = read_vector(
        population_gdb,
        layer=pop_layer_actual,
        columns=[pop_field],
        bbox=bbox,
    )

    if pop_candidates.empty:
        return pop_candidates, gpd.GeoDataFrame(geometry=[], crs=target_crs)

    pop_candidates = ensure_same_crs(pop_candidates, target_crs, "pop_grid")
    require_columns(pop_candidates, [pop_field], "pop_grid")

    # v2 fix: create _grid_fid directly from whatever index pyogrio returned.
    pop_candidates = add_stable_id_from_index(pop_candidates, "_grid_fid")

    all_blocks = pd.concat(
        [gdf[["geometry"]] for gdf in folder_block_layers.values() if not gdf.empty],
        ignore_index=True,
    )
    all_blocks = gpd.GeoDataFrame(all_blocks, geometry="geometry", crs=target_crs)
    all_blocks_union = geometry_union(all_blocks)

    pop_selected = pop_candidates[
        pop_candidates.geometry.intersects(all_blocks_union)
    ].copy()

    msg(f"  Candidate pop_grid cells read: {len(pop_candidates):,}")
    msg(f"  Selected pop_grid cells:       {len(pop_selected):,}")

    if pop_selected.empty:
        return pop_selected, gpd.GeoDataFrame(geometry=[], crs=target_crs)

    grid_bbox = tuple(float(v) for v in pop_selected.total_bounds)

    msg("  Reading candidate buildings_inside features from FileGDB...")
    msg(f"    selected grid bbox: {grid_bbox}")

    bldg_layer_actual = actual_layer_name(population_gdb, buildings_layer)

    try:
        buildings = read_vector(
            population_gdb,
            layer=bldg_layer_actual,
            columns=[],
            bbox=grid_bbox,
        )
    except Exception:
        buildings = read_vector(
            population_gdb,
            layer=bldg_layer_actual,
            bbox=grid_bbox,
        )

    buildings = ensure_same_crs(buildings, target_crs, "buildings_inside")

    grid_union = geometry_union(pop_selected)
    buildings = buildings[
        buildings.geometry.intersects(grid_union)
    ].copy()

    buildings = buildings[["geometry"]].copy()

    msg(f"  Buildings selected:            {len(buildings):,}")

    return pop_selected, buildings


def write_assigned_population_csv(csv_path, block_fid_field, block_ids, block_assigned_population):
    if csv_path.exists():
        msg("Deleting existing CSV:")
        msg(f"  {csv_path}")
        csv_path.unlink()

    msg("Writing final block population CSV:")
    msg(f"  {csv_path}")

    with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([block_fid_field, "assigned_population"])

        for block_id in sorted(block_ids):
            assigned_pop = safe_float(block_assigned_population.get(int(block_id), 0.0))
            writer.writerow([block_id, assigned_pop])


def write_grid_detail_csv(csv_path, block_fid_field, grid_detail_rows):
    if csv_path.exists():
        msg("Deleting existing CSV:")
        msg(f"  {csv_path}")
        csv_path.unlink()

    msg("Writing grid apportionment detail CSV:")
    msg(f"  {csv_path}")

    with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "FID_pop_grid_selection",
            "grid_code",
            block_fid_field,
            "group_built_area_m2",
            "total_built_area_m2_in_grid",
            "built_area_share",
            "apportioned_population",
            "note",
        ])

        for r in grid_detail_rows:
            writer.writerow([
                r["FID_pop_grid_selection"],
                r["grid_code"],
                r["block_fid"],
                r["group_built_area_m2"],
                r["total_built_area_m2_in_grid"],
                r["built_area_share"],
                r["apportioned_population"],
                r["note"],
            ])


# ============================================================
# CORE POPULATION ASSIGNMENT
# ============================================================

def calculate_population_for_block_layer(blocks_gdf, block_layer_name, pop_selected, buildings, block_folder, origin_membership):
    safe_block_name = sanitize_name(block_layer_name)
    k_suffix = get_k_suffix(block_layer_name)

    msg("")
    msg("------------------------------------------------------------")
    msg(f"Processing block layer: {block_layer_name}", also_console=True)
    msg(f"  k suffix: {k_suffix}")
    msg("------------------------------------------------------------")

    assigned_pop_csv = block_folder / f"{safe_block_name}_assigned_population.csv"
    grid_detail_csv = block_folder / f"{safe_block_name}_grid_apportionment_detail.csv"

    blocks = blocks_gdf.copy()

    if blocks.empty:
        blocks[population_field] = []
        blocks["hetero_orig"] = origin_membership["hetero_orig"]
        blocks["large_pop_orig"] = origin_membership["large_pop_orig"]
        write_assigned_population_csv(assigned_pop_csv, "block_fid", [], {})
        write_grid_detail_csv(grid_detail_csv, "block_fid", [])
        return blocks, {
            "block_layer": block_layer_name,
            "k": k_suffix,
            "selected_grid_cells": 0,
            "identity_features": None,
            "building_intersections": 0,
            "block_features_updated": 0,
            "total_grid_population_seen": 0.0,
            "total_population_assigned_to_blocks": 0.0,
            "status": "empty block layer",
        }

    if not is_projected_crs(blocks):
        msg("  WARNING: block layer CRS does not appear to be projected.")
        msg("  Area calculations may not be in square meters.")

    blocks = make_valid_if_requested(blocks, "blocks")
    pop_selected = make_valid_if_requested(pop_selected, "pop_grid")
    buildings = make_valid_if_requested(buildings, "buildings")

    block_union = geometry_union(blocks[["geometry"]])

    pop_for_layer = pop_selected[
        pop_selected.geometry.intersects(block_union)
    ].copy()

    selected_count = len(pop_for_layer)

    msg(f"Selected pop_grid cells for this layer: {selected_count:,}")

    def finish_zero(status):
        blocks[population_field] = 0.0
        blocks["hetero_orig"] = origin_membership["hetero_orig"]
        blocks["large_pop_orig"] = origin_membership["large_pop_orig"]
        write_assigned_population_csv(
            assigned_pop_csv,
            block_fid_field="block_fid",
            block_ids=list(blocks["_block_fid"]),
            block_assigned_population={},
        )
        write_grid_detail_csv(
            grid_detail_csv,
            block_fid_field="block_fid",
            grid_detail_rows=[],
        )
        return blocks, {
            "block_layer": block_layer_name,
            "k": k_suffix,
            "selected_grid_cells": selected_count,
            "identity_features": None,
            "building_intersections": 0,
            "block_features_updated": len(blocks),
            "total_grid_population_seen": 0.0,
            "total_population_assigned_to_blocks": 0.0,
            "status": status,
        }

    if selected_count == 0:
        msg("WARNING: No pop_grid cells selected. Population will be set to 0.")
        return finish_zero("no pop_grid cells selected")

    layer_grid_union = geometry_union(pop_for_layer)
    buildings_for_layer = buildings[
        buildings.geometry.intersects(layer_grid_union)
    ].copy()

    if buildings_for_layer.empty:
        msg("WARNING: No buildings intersect selected pop_grid cells. Population will be set to 0.")
        return finish_zero("no buildings in selected pop_grid cells")

    msg("Running overlay: selected pop_grid cells ∩ buildings...")

    grid_bldg = gpd.overlay(
        pop_for_layer[["_grid_fid", pop_field, "geometry"]],
        buildings_for_layer[["geometry"]],
        how="intersection",
        keep_geom_type=False,
    )

    if grid_bldg.empty:
        msg("WARNING: Grid-building overlay produced no features. Population will be set to 0.")
        return finish_zero("no grid-building intersections")

    grid_bldg[area_field] = grid_bldg.geometry.area
    grid_bldg = grid_bldg[grid_bldg[area_field] > area_tolerance].copy()

    building_intersections_count = len(grid_bldg)

    msg(f"Grid-building intersection features: {building_intersections_count:,}")

    if grid_bldg.empty:
        return finish_zero("all grid-building intersections had zero area")

    grid_total_built_area = (
        grid_bldg.groupby("_grid_fid", dropna=False)[area_field]
        .sum()
        .to_dict()
    )

    grid_population = (
        grid_bldg.groupby("_grid_fid", dropna=False)[pop_field]
        .first()
        .apply(safe_float)
        .to_dict()
    )

    msg("Running overlay: grid-building pieces ∩ block polygons...")

    block_bldg = gpd.overlay(
        grid_bldg[["_grid_fid", "geometry"]],
        blocks[["_block_fid", "geometry"]],
        how="intersection",
        keep_geom_type=False,
    )

    if block_bldg.empty:
        block_bldg[area_field] = []
    else:
        block_bldg[area_field] = block_bldg.geometry.area
        block_bldg = block_bldg[block_bldg[area_field] > area_tolerance].copy()

    msg(f"Grid-building-block intersection features: {len(block_bldg):,}")

    if block_bldg.empty:
        grid_block_built_area = pd.DataFrame(columns=["_grid_fid", "_block_fid", area_field])
    else:
        grid_block_built_area = (
            block_bldg.groupby(["_grid_fid", "_block_fid"], dropna=False)[area_field]
            .sum()
            .reset_index()
        )

    msg("Apportioning grid-cell population to block features...")

    block_assigned_population = defaultdict(float)
    grid_detail_rows = []

    inside_area_by_grid = defaultdict(dict)
    inside_total_by_grid = defaultdict(float)

    for _, r in grid_block_built_area.iterrows():
        grid_id = r["_grid_fid"]
        block_fid = int(r["_block_fid"])
        group_area = safe_float(r[area_field])
        inside_area_by_grid[grid_id][block_fid] = group_area
        inside_total_by_grid[grid_id] += group_area

    zero_area_grid_count = 0

    for grid_id in sorted(grid_total_built_area.keys(), key=lambda x: str(x)):
        total_area = safe_float(grid_total_built_area[grid_id])
        grid_pop = safe_float(grid_population.get(grid_id, 0.0))

        if total_area <= 0:
            zero_area_grid_count += 1
            grid_detail_rows.append({
                "FID_pop_grid_selection": grid_id,
                "grid_code": grid_pop,
                "block_fid": None,
                "group_built_area_m2": 0.0,
                "total_built_area_m2_in_grid": 0.0,
                "built_area_share": 0.0,
                "apportioned_population": 0.0,
                "note": "zero total built area in grid",
            })
            continue

        for block_fid, group_area in sorted(inside_area_by_grid.get(grid_id, {}).items()):
            built_area_share = group_area / total_area
            apportioned_pop = grid_pop * built_area_share

            block_assigned_population[block_fid] += apportioned_pop

            grid_detail_rows.append({
                "FID_pop_grid_selection": grid_id,
                "grid_code": grid_pop,
                "block_fid": block_fid,
                "group_built_area_m2": group_area,
                "total_built_area_m2_in_grid": total_area,
                "built_area_share": built_area_share,
                "apportioned_population": apportioned_pop,
                "note": "inside block",
            })

        outside_area = total_area - safe_float(inside_total_by_grid.get(grid_id, 0.0))

        if outside_area > area_tolerance:
            outside_share = outside_area / total_area
            outside_pop = grid_pop * outside_share

            grid_detail_rows.append({
                "FID_pop_grid_selection": grid_id,
                "grid_code": grid_pop,
                "block_fid": -1,
                "group_built_area_m2": outside_area,
                "total_built_area_m2_in_grid": total_area,
                "built_area_share": outside_share,
                "apportioned_population": outside_pop,
                "note": "outside blocks",
            })

    msg(f"Grid cells with zero total built area: {zero_area_grid_count:,}")
    msg(f"Block features receiving population: {len(block_assigned_population):,}")

    write_assigned_population_csv(
        assigned_pop_csv,
        block_fid_field="block_fid",
        block_ids=list(blocks["_block_fid"]),
        block_assigned_population=block_assigned_population,
    )

    write_grid_detail_csv(
        grid_detail_csv,
        block_fid_field="block_fid",
        grid_detail_rows=grid_detail_rows,
    )

    blocks[population_field] = blocks["_block_fid"].map(
        lambda x: float(block_assigned_population.get(int(x), 0.0))
    )

    # Add original-source membership fields to the populated block layer.
    blocks["hetero_orig"] = origin_membership["hetero_orig"]
    blocks["large_pop_orig"] = origin_membership["large_pop_orig"]

    updated_count = len(blocks)
    zero_count = int((blocks[population_field] == 0).sum())

    total_grid_population_seen = sum(safe_float(v) for v in grid_population.values())
    total_population_assigned_to_blocks = sum(safe_float(v) for v in block_assigned_population.values())

    msg("Layer done.")
    msg("Summary:")
    msg(f"  Block layer:                         {block_layer_name}")
    msg(f"  Selected grid cells:                 {selected_count:,}")
    msg(f"  Identity features:                   not created in GeoPandas version")
    msg(f"  Grid-building intersections:         {building_intersections_count:,}")
    msg(f"  Grid-building-block intersections:   {len(block_bldg):,}")
    msg(f"  Block features updated:              {updated_count:,}")
    msg(f"  Block features assigned zero pop:    {zero_count:,}")
    msg(f"  Total grid population represented:   {total_grid_population_seen}")
    msg(f"  Total population assigned to blocks: {total_population_assigned_to_blocks}")

    return blocks, {
        "block_layer": block_layer_name,
        "k": k_suffix,
        "selected_grid_cells": selected_count,
        "identity_features": None,
        "building_intersections": building_intersections_count,
        "block_features_updated": updated_count,
        "total_grid_population_seen": total_grid_population_seen,
        "total_population_assigned_to_blocks": total_population_assigned_to_blocks,
        "status": "success",
    }


# ============================================================
# MAIN
# ============================================================

def main():
    if log_file.exists():
        log_file.unlink()

    msg("Starting no-ArcPy population assignment", also_console=True)
    msg(f"Block folder root: {large_pop_blocks_folder}", also_console=True)
    msg(f"Population GDB:    {population_gdb}", also_console=True)
    msg(f"Log file:          {log_file}", also_console=True)
    msg("", also_console=True)

    if not large_pop_blocks_folder.is_dir():
        raise FileNotFoundError(
            "large_pop_blocks_folder does not exist:\n"
            f"{large_pop_blocks_folder}"
        )

    if not population_gdb.exists():
        raise FileNotFoundError(
            "Population geodatabase does not exist:\n"
            f"{population_gdb}"
        )

    if not source_blocks_gdb.exists():
        raise FileNotFoundError(
            "Source blocks geodatabase does not exist:\n"
            f"{source_blocks_gdb}"
        )

    if not layer_exists(population_gdb, pop_grid_layer):
        raise FileNotFoundError(
            f"Population grid layer '{pop_grid_layer}' does not exist in:\n"
            f"{population_gdb}"
        )

    if not layer_exists(population_gdb, buildings_layer):
        raise FileNotFoundError(
            f"Buildings layer '{buildings_layer}' does not exist in:\n"
            f"{population_gdb}"
        )

    source_lookup = read_source_block_membership_lookup()

    block_folders = list(iter_block_folders(large_pop_blocks_folder))

    msg("Block folders found:")
    msg(f"  {len(block_folders)}")

    overall_summary = []

    for block_folder in block_folders:
        block_folder_name = block_folder.name

        msg("")
        msg("============================================================")
        msg(f"Block folder: {block_folder_name}", also_console=True)
        msg("============================================================")

        new_blocks_gpkg = block_folder / new_blocks_gpkg_name
        populated_blocks_gpkg = block_folder / populated_blocks_gpkg_name

        if not new_blocks_gpkg.exists():
            msg("Skipping folder because new_blocks.gpkg does not exist:")
            msg(f"  {new_blocks_gpkg}")
            continue

        try:
            block_layer_names = get_blk_layers(new_blocks_gpkg)

            if not block_layer_names:
                msg("No blk_* layers found in:")
                msg(f"  {new_blocks_gpkg}")
                continue

            msg("Block layers found:")
            for layer_name in block_layer_names:
                msg(f"  {layer_name}")

            origin_membership = get_origin_membership_for_folder(
                block_folder_name,
                source_lookup,
            )

            if overwrite_outputs and populated_blocks_gpkg.exists():
                msg("Deleting existing populated output GeoPackage:")
                msg(f"  {populated_blocks_gpkg}")
                populated_blocks_gpkg.unlink()

            block_layers = read_block_layers_for_folder(
                new_blocks_gpkg,
                block_layer_names,
            )

            pop_selected, buildings = read_population_candidates(block_layers)

            for layer_name in block_layer_names:
                try:
                    updated_gdf, result = calculate_population_for_block_layer(
                        blocks_gdf=block_layers[layer_name],
                        block_layer_name=layer_name,
                        pop_selected=pop_selected,
                        buildings=buildings,
                        block_folder=block_folder,
                        origin_membership=origin_membership,
                    )

                    if "_block_fid" in updated_gdf.columns:
                        updated_gdf_to_write = updated_gdf.drop(columns=["_block_fid"]).copy()
                    else:
                        updated_gdf_to_write = updated_gdf.copy()

                    write_gpkg_layer(
                        updated_gdf_to_write,
                        populated_blocks_gpkg,
                        layer_name,
                    )

                    msg("Populated block layer written to:", also_console=True)
                    msg(f"  {populated_blocks_gpkg} | {layer_name}")

                    result["block_folder"] = block_folder_name
                    overall_summary.append(result)

                except Exception as e:
                    msg("")
                    msg("ERROR while processing:")
                    msg(f"  Folder: {block_folder_name}")
                    msg(f"  Layer:  {layer_name}")
                    msg(str(e))

                    overall_summary.append({
                        "block_folder": block_folder_name,
                        "block_layer": layer_name,
                        "k": get_k_suffix(layer_name),
                        "selected_grid_cells": None,
                        "identity_features": None,
                        "building_intersections": None,
                        "block_features_updated": None,
                        "total_grid_population_seen": None,
                        "total_population_assigned_to_blocks": None,
                        "status": f"ERROR: {str(e)}",
                    })

        except Exception:
            msg("")
            msg("FAILED on this block folder:")
            msg(traceback.format_exc())

            overall_summary.append({
                "block_folder": block_folder_name,
                "block_layer": None,
                "k": None,
                "selected_grid_cells": None,
                "identity_features": None,
                "building_intersections": None,
                "block_features_updated": None,
                "total_grid_population_seen": None,
                "total_population_assigned_to_blocks": None,
                "status": "ERROR at folder level",
            })

    summary_csv = large_pop_blocks_folder / "new_blocks_population_assignment_summary_no_arcpy.csv"

    if summary_csv.exists():
        msg("Deleting existing overall summary CSV:")
        msg(f"  {summary_csv}")
        summary_csv.unlink()

    msg("")
    msg("Writing overall summary CSV:")
    msg(f"  {summary_csv}")

    with open(summary_csv, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "block_folder",
                "block_layer",
                "k",
                "selected_grid_cells",
                "identity_features",
                "building_intersections",
                "block_features_updated",
                "total_grid_population_seen",
                "total_population_assigned_to_blocks",
                "status",
            ],
        )

        writer.writeheader()

        for row in overall_summary:
            writer.writerow(row)

    msg("")
    msg("All done.", also_console=True)
    msg(f"Layers processed: {len(overall_summary)}", also_console=True)
    msg("Summary written to:", also_console=True)
    msg(f"  {summary_csv}", also_console=True)


if __name__ == "__main__":
    main()
